# Step 11 — Measure it

*Step 11 of the AI in Industry lab*

---

## Read this before you run anything

You will score the system against ten questions whose correct answers you already know, then change one setting and score it again.

**What you should end up understanding:** How to tell whether a change made things better, with a number instead of an opinion. Almost nobody does this.

| | |
|---|---|
| **Cost** | 30 API calls - takes a few minutes |
| **Needs earlier steps?** | No. This notebook sets itself up. |
| **Safe to re-run?** | Yes, but see the warning — it is the most expensive notebook in the lab. |

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; This notebook makes about 30 API calls and takes several minutes. On a free key that is a large chunk of your quota.</b></div>

<div style="background:#ffebee;border-left:6px solid #c62828;padding:12px 16px;margin:10px 0;border-radius:4px"><b style="color:#c62828">&#9888; Run the scoring cell once, read the results properly, and only re-run it after you have deliberately changed something. The comparison between two runs is the whole point — a single score on its own tells you nothing.</b></div>

**Now run the Setup cell.** About 30 seconds. It works even if you skipped every earlier step.

In [ ]:
#@title Setup - run this first (about 30 seconds) { display-mode: "form" }
# Fetches the lab files, installs what is needed, reads your API key.
# Identical in every step notebook, so any step works on its own.
import os, sys, pathlib, subprocess

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if not pathlib.Path("rough").exists():
    print("Downloading the lab files ...")
    subprocess.run("git clone --depth 1 --quiet "
                   "https://github.com/coolMukul/rough.git rough", shell=True)
os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing (slow the first time) ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt", shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

if not os.environ.get("LLM_API_KEY"):
    print("\n  NO API KEY. Click the key icon on the left, add a secret named")
    print("  exactly LLM_API_KEY, paste your key from console.groq.com/keys,")
    print("  and turn ON 'Notebook access'. Then run this cell again.")
else:
    print(f"\nReady. Key ending ...{os.environ['LLM_API_KEY'][-4:]}")

Everyone demos on the three questions that work. **The ones who ship, measure.**

This is the step that gets you hired, and almost nobody does it.

In [ ]:
from labcore import corpus, tfidf, embed, grounded

chunks, texts, _ = corpus()
print("ready")

### The test set

Ten questions whose correct answers we already know, checked by hand against the PDF. Five categories, two each.

In [ ]:
# ===== EDIT ME, then run the cell below =====
EVAL = [
    ("What is the minimum attendance required in each course?", "75", "lookup"),
    ("How many credits are required for the B.Tech degree?", "160", "lookup"),
    ("I have 68% attendance because of placement drives. What happens?",
     ["condon", "10"], "two-docs"),
    ("I scored 18 out of 60 in formative assessment. What grade do I get?",
     ["R", "21"], "two-docs"),
    ("What are the hostel mess timings?", "REFUSE", "not-in-corpus"),
    ("How much is the tuition fee per semester?", "REFUSE", "not-in-corpus"),
    ("Can I get an exemption if I miss too many classes?", "condon", "paraphrased"),
    ("What do I need to score to not fail the internals?", "21", "paraphrased"),
    ("My attendance is exactly 75%. Am I eligible for the end sem?", "75", "edge"),
    ("My CGPA is exactly 7.0. What class do I get?", "distinction", "edge"),
]

print(f"{len(EVAL)} test questions")

Look at rows 5 and 6: **the correct answer is a refusal.** Refusing correctly scores a pass.

And rows 9 and 10 sit exactly on a boundary — *exactly* 75%, *exactly* 7.0. Those are the questions a real student actually worries about, and they are where wording matters more than numbers.

### The scorer

In [ ]:
def grade(answer, expected):
    low = answer.lower()
    if expected == "REFUSE":
        return "could not find" in low
    needed = expected if isinstance(expected, list) else [expected]
    return all(n.lower() in low for n in needed)


def score(ask, k, label):
    passed = 0
    print(f"\n{label}")
    for question, expected, category in EVAL:
        ok = grade(ask(question, k=k), expected)
        passed += ok
        print(f"  {'PASS' if ok else 'FAIL'}  [{category:13}] {question[:48]}")
    print(f"  --> {passed}/{len(EVAL)}")
    return passed


print("scorer defined - no API calls yet")

### Score one configuration

Each run of the cell below makes **10 API calls** and takes about half a minute.

In [ ]:
# ===== EDIT ME, then run the cell below =====
retriever = "tfidf"      # "tfidf" or "embed"
k = 1                    # how many clauses to retrieve

In [ ]:
r = tfidf(texts) if retriever == "tfidf" else embed(texts)
result = score(grounded(r), k, f"{retriever}, k={k}")

### Now change ONE thing and score again

Go back to the settings cell, change `k` to `6`, and run the scoring cell again.

**Write both numbers down.** The comparison is the entire point — a single score on its own tells you nothing.

---

## Read the failures, not just the score

Look at **which categories** failed.

If they cluster in `paraphrased`, your problem is retrieval and a bigger model will not help. If they cluster in `two-docs`, raise `k`. If `not-in-corpus` fails, your system is inventing and the system prompt needs work.

**That is a diagnosis with evidence behind it**, which is a completely different thing from a hunch.

---

## The sentence that gets you hired

> *"I built a RAG chatbot."* — everyone says this.
>
> *"I measured it at 60% on my own eval set, found the failures were retrieval not generation, and got it to 85%."* — almost nobody says this.

You have just done the second one in miniature. **Keep the numbers.**

---

## Now change it yourself

Add a question you can verify in your own regulations.

In [ ]:
# ===== EDIT ME, then run the cell below =====
EVAL = EVAL + [
    # (question, text that must appear in a correct answer, category)
    ("How long do I have to complete the degree?", "seven", "lookup"),
]

print(f"{len(EVAL)} questions now")

Then re-run the scoring cell above.

**Ten questions of your own, on documents you care about, is a weekend project and an interview answer.**

---

### Done with step 11

Open the next step's notebook. If something here did not work, **do not stop to debug it** — every step sets itself up from scratch, so the next one will still run.